# backward-on-scalar-loss — worked example 3: backward() accumulates into .grad — zero it between calls

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `backward-on-scalar-loss`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

`.backward()` **adds** the new gradient to whatever is already in `.grad` rather than overwriting it. Calling `.backward()` twice without clearing doubles the gradient. To get the correct gradient on the second call you must reset with `w.grad.zero_()` (or set it to `None`).

## Worked solution

1. **First backward.** Compute `loss1 = ((w * x - y) ** 2).mean()` and call `loss1.backward()`. This populates `w.grad` with `g = dL/dw`.
2. **The accumulation trap.** If we immediately recompute the loss and call `.backward()` again, autograd *accumulates*: `w.grad` becomes `g + g = 2g`. This is by design — it lets you sum gradients across mini-batches — but it is wrong if you wanted a fresh gradient.
3. **Zero the grad.** Call `w.grad.zero_()` to reset the leaf's gradient buffer to all zeros in place. (Optimizers expose this as `optimizer.zero_grad()`.)
4. **Second backward, fresh graph.** We recompute `loss2` (the forward graph is consumed by each backward, so we build it again) and call `loss2.backward()`. Now `w.grad` holds exactly `g` again, not `2g`.
5. **Verify.** We capture the grad right after the first backward and again after zero+second backward; they should be equal, proving the zero reset worked.

In [ ]:
def backward_with_zero(w, x, y):
    loss1 = ((w * x - y) ** 2).mean()
    loss1.backward()
    grad_first = w.grad.clone()
    w.grad.zero_()
    loss2 = ((w * x - y) ** 2).mean()
    loss2.backward()
    grad_second = w.grad.clone()
    return grad_first, grad_second

t.manual_seed(0)
w = t.tensor([0.8], requires_grad=True)
x = t.tensor([1.0, 2.0, 3.0])
y = t.tensor([1.0, 1.5, 2.5])
g1, g2 = backward_with_zero(w, x, y)
print('grad after 1st backward:', g1)
print('grad after zero + 2nd backward:', g2)
print('equal (zero_ worked):', t.allclose(g1, g2))